# The Agent Evaluation Challenge (Why Traditional Software Testing and LLM Evals Fail)
As you transition students from building traditional software or standard single-prompt LLM applications to building autonomous agents, they will quickly realize that everything they know about testing breaks down.

In this topic, we examine why traditional software unit testing and standard LLM evaluation techniques fail for agents, and introduce the mental model required to evaluate non-deterministic, multi-step systems.

## 1. Why Traditional Software Testing Fails for Agents
Traditional software engineering relies on deterministic inputs and outputs. If you write a Python function add(a, b), you expect add(2, 3) to return 5 every single time.

In [ ]:
# Traditional Unit Test (Deterministic)
def test_add():
    assert add(2, 3) == 5  # Passes 100% of the time if logic is correct

### Why this breaks for Agents:

**Non-Determinism (Stochasticity):** LLMs sample tokens probabilistically. Running the exact same prompt and task twice might yield different reasoning paths, phrasing, or tool-calling sequences.

**Open-Ended Outputs:** Unlike a function that returns an integer or a specific JSON schema, an agent often generates natural language responses. Two completely different sentences can both be 100% correct answers to the same user query.

**Environment Dependency:** Agents interact with external state (APIs, databases, file systems, live web pages). If a database record changes or a third-party API goes down, the agent's environment changes, leading to different execution trajectories for the same code.

## 2. Why Standard LLM Evals (Prompt-Response) Fail for Agents
Standard LLM evaluation tools (like BLEU, ROUGE, or simple LLM-as-a-judge prompt grading) evaluate a single turn: Prompt ──► LLM ──► Response.

[Standard LLM Eval]
[User Prompt] ──► [LLM Call] ──► [Single Output Response] (Scored once)

### Why this breaks for Agents:

Multi-Step Trajectories: Agents don't just generate an answer; they act. An agentic workflow looks like a loop: Thought ──► Tool Call ──► Observation ──► Next Thought ──► Final Answer.

Compounding Errors: If an agent makes a tiny mistake on Step 1 (e.g., passing a slightly malformed argument to a search tool), that error cascades. Step 2 receives garbage data, Step 3 hallucinates a correction, and the final output is completely wrong.

Scoring the Journey vs. The Destination: In standard LLMs, you only care about the destination (the final response). In agents, how the agent arrived at the destination (efficiency, safety, tool selection accuracy) matters just as much as the final answer. An agent that arrives at the correct answer by stumbling through 50 infinite loops and crashing 3 tools is a failure, even if the final sentence is correct.

[Agent Trajectory Eval]
[User Prompt] ──► [Thought 1] ──► [Tool Call A] ──► [Observation A] ──► [Thought 2] ... 
                        │                 │                  │
                    (Scored)          (Scored)           (Scored)  <-- Every step must be evaluated!

## 3. The Three Core Dimensions of Agent Evaluation
To evaluate an agent successfully, your test framework must measure three distinct layers:

**Path Efficiency (Trajectory):** Did the agent take a direct, logical route to solve the problem, or did it wander through redundant steps and circular reasoning?

**Tool Correctness:** Did the agent select the right tool for the subtask? Did it adhere strictly to the JSON schema requirements when passing arguments?

**Task Completion & Safety:** Did the agent achieve the user's ultimate goal without violating safety guardrails or hallucinating unverified facts?

## 4. Key Takeaways
**Mindset Shift:** Teach students to stop thinking in terms of assert result == expected and start thinking in terms of distributions, trajectories, and rubrics.

**Observability First:** Emphasize that you cannot evaluate what you cannot see. Before writing an evaluation script, students must implement end-to-end tracing (recording every thought, tool call, and state variable) so they can replay failing runs.